# EDA Dataset Superstore — Panduan Lengkap

**Sub-CPMK:** **P4** — EDA univariat/bivariat; visualisasi dengan Matplotlib & Seaborn.

Notebook ini adalah **tutorial mandiri** (melengkapi `minggu_05.ipynb`): eksplorasi penjualan ritel fiktif Tableau **Sample Superstore**, fokus **`Profit`** (bisa negatif), diskon, dan dimensi produk/geografi.

**Konteks data:** [Kaggle — Sample Superstore](https://www.kaggle.com/datasets/konstantinognev/sample-superstorecsv/data) — **9.994** baris transaksi, variabel bisnis inti **Profit**, **Sales**, **Discount**, **Quantity**.

**Sumber data:** mirror GitHub (CSV klasik 9.994 baris). **Unduhan pertama membutuhkan internet.**

```python
# Alternatif offline: salin SampleSuperstore.csv ke data/ dan ganti read_csv
```

**Pertanyaan analitik:**
- Kategori/sub-kategori mana paling menguntungkan atau merugi?
- Region dan Segment mana dominan profit vs sales?
- Bagaimana hubungan **Discount** dengan **Profit**?
- Di mana order rugi (`Profit < 0`) terkonsentrasi?

**Referensi:**
- [Analytics Vidhya — EDA SuperStore (2022)](https://www.analyticsvidhya.com/blog/2022/03/eda-on-superstore-dataset-using-python/)
- [33rd Square — EDA Superstore](https://33rdsquare.com/eda-on-superstore-dataset-using-python/)
- [GitHub — shekhs/SampleSuperstore-EDA](https://github.com/shekhs/SampleSuperstore-EDA)
- [leonism/sample-superstore](https://github.com/leonism/sample-superstore) (varian extended)
- Modul PDF: `modul-05.tex` (Minggu 5); regresi: `minggu_07.ipynb`


## 0. Kerangka CRISP-DM dan kamus fitur

Fase **Data Understanding** (CRISP-DM): pahami kolom sebelum memplot. EDA **iteratif** — temuan mengarahkan cleaning (`minggu_03`), encoding (`minggu_04`), regresi (`minggu_07`).

| Kolom | Arti singkat | Tipe |
|-------|----------------|------|
| `Ship_Mode` | Metode pengiriman (Standard Class, First Class, …) | kategorikal |
| `Segment` | Segmen pelanggan: Consumer, Corporate, Home Office | kategorikal |
| `Country` | Negara (semua United States) | kategorikal |
| `City`, `State` | Lokasi pelanggan | kategorikal |
| `Postal_Code` | Kode pos (granular) | kategorikal |
| `Region` | Central, East, South, West | kategorikal |
| `Category` | Furniture, Office Supplies, Technology | kategorikal |
| `Sub_Category` | Sub-kategori produk (17 level) | kategorikal |
| `Sales` | Pendapatan baris order ($) | numerik |
| `Quantity` | Jumlah unit | numerik |
| `Discount` | Diskon (proporsi 0–1) | numerik |
| `Profit` | Laba baris order ($), **bisa negatif** | numerik |

**Kolom turunan (§2):** `profit_margin = Profit / Sales`, `is_loss = Profit < 0`.

**Catatan:** File mentah juga berisi `Order_ID`, `Order_Date`, dll.; EDA inti memakai subset 11 kolom setelah drop `Country`/`Postal_Code`.


## 1. Persiapan lingkungan

Import pustaka dan atur tema Seaborn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
%matplotlib inline

**Memuat data.** Encoding `latin-1` diperlukan untuk mirror CSV ini; normalisasi nama kolom ke `snake_case`.

In [ ]:
URL = (
    "https://raw.githubusercontent.com/sumit0072/Superstore-Data-Analysis/"
    "main/Sample%20-%20Superstore.csv"
)
df = pd.read_csv(URL, encoding="latin-1")
df.columns = df.columns.str.replace(" ", "_").str.replace("-", "_")

print("Bentuk data mentah:", df.shape)
df.head()

**Inspeksi awal.** `info()` dan `describe()` pada data mentah.

In [ ]:
display(df.tail())
df.info()
display(df.describe())
display(df.describe(include="object"))

### 1b. Audit kualitas data

`sample` untuk spot-check; cek duplikat; bentuk `df_eda` untuk analisis (drop kolom redundan/granular).

In [ ]:
print("Cuplikan acak 20 baris:")
display(df.sample(20, random_state=RANDOM_STATE))

print("\nBaris duplikat penuh:", df.duplicated().sum())
print("Negara unik:", df["Country"].unique())
print("Jumlah State:", df["State"].nunique())
print("Jumlah Sub_Category:", df["Sub_Category"].nunique())

# Subset analisis: buang Country (semua US) dan Postal_Code (terlalu granular)
drop_cols = ["Country", "Postal_Code", "Row_ID", "Order_ID", "Order_Date", "Ship_Date",
             "Customer_ID", "Customer_Name", "Product_ID", "Product_Name"]
df_eda = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()
print("\nBentuk df_eda:", df_eda.shape)

**Interpretasi (audit):** Dataset klasik **9.994** baris tanpa missing pada kolom inti; semua transaksi AS. Kolom order/pelanggan disimpan di `df` mentah; `df_eda` fokus dimensi produk–geografi–finansial.

## 2. EDA tabular (Part I)

Ringkasan numerik, frekuensi kategori, dan korelasi terhadap **Profit**.

In [ ]:
print("Missing per kolom (df_eda):")
print(df_eda.isna().sum())

for col in ["Segment", "Category", "Region", "Ship_Mode"]:
    print(f"\n--- {col} ---")
    print(df_eda[col].value_counts())

In [ ]:
n = len(df_eda)
n_loss = (df_eda["Profit"] < 0).sum()
print(f"Total Sales: ${df_eda['Sales'].sum():,.2f}")
print(f"Total Profit: ${df_eda['Profit'].sum():,.2f}")
print(f"Baris rugi (Profit < 0): {n_loss} ({100*n_loss/n:.1f}%)")

num_cols = ["Sales", "Quantity", "Discount", "Profit"]
summary = df_eda[num_cols].agg(["mean", "median", "std", "min", "max"])
display(summary.round(2))

**Mean vs median:** Jika mean jauh dari median → skew/outlier (umum pada `Sales` dan `Discount`).

In [ ]:
# profit_margin: hindari pembagian nol
zero_sales = (df_eda["Sales"] == 0).sum()
print("Baris dengan Sales = 0:", zero_sales)
df_eda["profit_margin"] = np.where(df_eda["Sales"] != 0, df_eda["Profit"] / df_eda["Sales"], np.nan)
df_eda["is_loss"] = df_eda["Profit"] < 0

corr = df_eda[num_cols + ["profit_margin"]].corr()
display(corr.round(3))
print("\nKorelasi dengan Profit (diurutkan):")
print(corr["Profit"].drop("Profit").sort_values(key=abs, ascending=False).round(3))

**Interpretasi (tabel):** Sekitar **18,7%** baris merugi; **Discount** berkorelasi negatif dengan **Profit**; **Sales** dan **Profit** berkorelasi positif moderat. Mean `Sales` > median → skew kanan.

## 3. EDA univariat (Part II — distribusi)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.flat, num_cols):
    sns.histplot(df_eda[col], kde=True, ax=ax)
    ax.set_title(f"Distribusi {col}")
    ax.set_xlabel(col)
plt.suptitle("Histogram + KDE variabel numerik", y=1.02)
plt.tight_layout()
plt.show()

**Interpretasi:** `Profit` terpusat di sekitar 0 dengan massa negatif; `Discount` skew kanan (banyak order diskon rendah); `Quantity` dominan 1–2 unit.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.flat, ["Ship_Mode", "Segment", "Category", "Region"]):
    order = df_eda[col].value_counts().index
    sns.countplot(data=df_eda, x=col, order=order, ax=ax)
    ax.set_title(f"Frekuensi {col}")
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 4. EDA bivariat — Profit, Sales, dan dimensi bisnis

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df_eda[num_cols].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Heatmap korelasi Pearson")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df_eda, x="Category", y="Profit", ax=axes[0])
axes[0].set_title("Profit per Category")
axes[0].tick_params(axis="x", rotation=20)
sns.boxplot(data=df_eda, x="Region", y="Profit", ax=axes[1])
axes[1].set_title("Profit per Region")
plt.tight_layout()
plt.show()

**Interpretasi:** **Technology** median profit tertinggi; **South** punya lebih banyak outlier rugi dibanding region lain ([33rd Square](https://33rdsquare.com/eda-on-superstore-dataset-using-python/)).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.violinplot(data=df_eda, x="Segment", y="Sales", ax=axes[0])
axes[0].set_title("Sales per Segment")
sns.violinplot(data=df_eda, x="Category", y="Sales", ax=axes[1])
axes[1].set_title("Sales per Category")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df_eda, x="Sales", y="Profit", hue="Category", alpha=0.5, size="Quantity",
    sizes=(20, 200), legend="brief",
)
plt.title("Sales vs Profit (hue: Category)")
plt.xlabel("Sales")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(data=df_eda, x="Discount", y="Profit", scatter_kws={"alpha": 0.3}, line_kws={"color": "crimson"})
plt.title("Discount vs Profit (dengan garis regresi linear)")
plt.xlabel("Discount")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()

**Interpretasi:** Diskon tinggi cenderung berkaitan dengan **Profit** lebih rendah — kebijakan diskon perlu diuji per sub-kategori, bukan global.

In [ ]:
sub_profit = (
    df_eda.groupby("Sub_Category", observed=True)["Profit"]
    .mean()
    .sort_values()
)
plt.figure(figsize=(10, 5))
sns.barplot(x=sub_profit.values, y=sub_profit.index, hue=sub_profit.index, palette="RdYlGn", legend=False)
plt.title("Rata-rata Profit per Sub_Category")
plt.xlabel("Mean Profit ($)")
plt.ylabel("Sub_Category")
plt.tight_layout()
plt.show()

In [ ]:
pivot_rc = df_eda.pivot_table(
    index="Region", columns="Category", values="Profit", aggfunc="mean", observed=True
)
print("Mean Profit — Region × Category:")
display(pivot_rc.round(2))

plt.figure(figsize=(7, 4))
sns.heatmap(pivot_rc, annot=True, fmt=".1f", cmap="YlGnBu", center=0)
plt.title("Heatmap mean Profit: Region × Category")
plt.xlabel("Category")
plt.ylabel("Region")
plt.tight_layout()
plt.show()

**Interpretasi (Region×Category):** **South** + **Office Supplies** / **Furniture** sering profit rata-rata rendah; **Technology** relatif kuat di semua region.

## 5. Analisis order rugi (`loss_df`)

Mengikuti [Analytics Vidhya](https://www.analyticsvidhya.com/blog/2022/03/eda-on-superstore-dataset-using-python/): fokus baris `Profit < 0`.

In [ ]:
loss_df = df_eda[df_eda["Profit"] < 0].copy()
total_loss = -loss_df["Profit"].sum()
print("Bentuk loss_df:", loss_df.shape)
print(f"Total kerugian (agregat): ${total_loss:,.2f}")
display(loss_df[num_cols].describe().round(2))

In [ ]:
print("Agregat rugi menurut Segment:")
display(loss_df.groupby("Segment", observed=True)[["Sales", "Discount", "Profit"]].sum().round(2))

print("\nAgregat rugi menurut Sub_Category (Profit terendah):")
sub_loss = loss_df.groupby("Sub_Category", observed=True)["Profit"].sum().sort_values()
display(sub_loss.head(10).round(2))

In [ ]:
print("Top 10 City dengan total Profit terendah (rugi terbesar):")
city_loss = loss_df.groupby("City", observed=True)["Profit"].sum().sort_values().head(10)
display(city_loss.round(2))

print("\nTop 10 State dengan total Profit terendah:")
state_loss = loss_df.groupby("State", observed=True)["Profit"].sum().sort_values().head(10)
display(state_loss.round(2))

In [ ]:
loss_sub_counts = loss_df["Sub_Category"].value_counts().head(8)
plt.figure(figsize=(9, 4))
sns.barplot(x=loss_sub_counts.index, y=loss_sub_counts.values, hue=loss_sub_counts.index, legend=False)
plt.title("Frekuensi baris rugi per Sub_Category (top 8)")
plt.xlabel("Sub_Category")
plt.ylabel("Jumlah baris rugi")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

**Interpretasi (rugi):** Total kerugian agregat ~**$156 ribu** (orde Analytics Vidhya). Sub-kategori seperti **Binders**, **Tables**, **Machines** sering muncul di `loss_df`; diskon tinggi pada baris rugi — selaraskan kebijakan promo dengan margin.

## 6. Segmentasi per Category

Profit rata-rata per `Sub_Category` dalam tiap `Category`.

In [ ]:
categories = ["Furniture", "Office Supplies", "Technology"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, cat in zip(axes, categories):
    sub = df_eda[df_eda["Category"] == cat]
    means = sub.groupby("Sub_Category", observed=True)["Profit"].mean().sort_values()
    sns.barplot(x=means.values, y=means.index, ax=ax, hue=means.index, legend=False)
    ax.set_title(cat)
    ax.set_xlabel("Mean Profit ($)")
    ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)
plt.suptitle("Mean Profit per Sub_Category (per Category)", y=1.05)
plt.tight_layout()
plt.show()

**Interpretasi:** **Bookcases**, **Supplies** (Office Supplies) mean profit negatif; **Copiers** dan **Phones** relatif kuat — pertimbangkan kurangi promo pada sub-kategori defisit.

## 7. Multivariat dan cuplikan encoding

Pairplot pada sampel (performa) dan heatmap setelah `get_dummies` (menuju `minggu_04`).

In [ ]:
pair_cols = ["Sales", "Quantity", "Discount", "Profit", "Category"]
sample_pp = df_eda[pair_cols].dropna()
if len(sample_pp) > 2000:
    sample_pp = sample_pp.sample(2000, random_state=RANDOM_STATE)
sns.pairplot(sample_pp, hue="Category", corner=True, plot_kws={"alpha": 0.4})
plt.show()

In [ ]:
# Sneak peek encoding — bukan pipeline final
df_enc = pd.get_dummies(
    df_eda.drop(columns=["City", "State"], errors="ignore"),
    columns=["Ship_Mode", "Segment", "Region", "Category", "Sub_Category"],
    drop_first=True,
)
top_corr = df_enc.corr(numeric_only=True)["Profit"].drop("Profit").abs().sort_values(ascending=False).head(10)
print("Top |korelasi| dengan Profit (setelah one-hot):")
display(top_corr.round(3))

plt.figure(figsize=(10, 8))
sns.heatmap(df_enc.corr(numeric_only=True), cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Heatmap korelasi — fitur encoded (cuplikan)")
plt.tight_layout()
plt.show()

**Catatan:** Korelasi encoded ≠ kausalitas; banyak kategori → interpretasi hati-hati terhadap multikolinearitas.

## 8. Ringkasan temuan, bias, dan hipotesis pemodelan

### Ringkasan temuan (EDA)

1. **Skala:** 9.994 transaksi; total **Profit** positif secara agregat tetapi **~18,7%** baris individual merugi.
2. **Kerugian:** Total kerugian pada baris `Profit < 0` ~**$156k**; terkonsentrasi pada sub-kategori tertentu (Binders, Tables, …).
3. **Kategori:** **Technology** profit rata-rata tertinggi; **Office Supplies** margin tipis.
4. **Region:** **South** underperform pada visual boxplot/heatmap; perlu strategi regional.
5. **Diskon:** Korelasi negatif **Discount–Profit**; diskon agresif sering pada baris rugi.
6. **Segment:** **Consumer** volume terbesar; profitabilitas per segment berbeda pada agregat rugi.
7. **Sales vs Profit:** Pemisahan tidak linear — perlu fitur kategori dan diskon dalam model.

### Bias dan limitasi

- Data **fiktif** Tableau; tidak merepresentasikan satu perusahaan riil.
- Kolom **tanggal** (`Order_Date`) tidak dieksplorasi mendalam di notebook ini — tidak ada analisis musiman/time series.
- Agregat baris order ≠ profitabilitas pelanggan jangka panjang.
- EDA **iteratif** (CRISP-DM): ulangi setelah baseline model jika residual bermasalah per region.

### Hipotesis pemodelan

- **Regresi** `Profit` atau `Sales` dari `Discount`, `Quantity`, dummy `Category`/`Region`/`Segment` (`minggu_07.ipynb`, RMSE/MAE).
- **Klasifikasi** opsional: prediksi `is_loss` untuk flag order defisit (logistic regression / pohon).
- Transformasi **log1p(Sales)** atau clip outlier dapat diuji pada minggu cleaning.

### Langkah berikutnya

1. `minggu_03.ipynb` — outlier, transformasi
2. `minggu_04.ipynb` — encoding & scaling
3. `minggu_07.ipynb` — regresi & evaluasi

Jalankan **Kernel → Restart & Run All** pada notebook ini.
